In [ ]:
from pathlib import Path
import os

IMAGES_PATH = Path("../resources/cloud-images/CCSN_v2")

def index_labeled_images(images_path=IMAGES_PATH):
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images

    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            labeled_images[img_path.name] = {
                "label": cloud_dir.name,
                "path": str(img_path)
            }

    return labeled_images

In [ ]:
labeled_images = index_labeled_images()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

def extract_labels(labeled_images):
    labels = []
    paths = []
    for (i, image_name) in enumerate(labeled_images):
        path = labeled_images[image_name]['path']
        paths.append(path)
        label = labeled_images[image_name]['label']
        labels.append(label)

    return paths, np.array(labels)

In [ ]:
paths, labels = extract_labels(labeled_images)

In [ ]:
print(labels)

In [ ]:
print(paths)

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
from torch.utils.data import DataLoader
import torchvision
import torch.nn as nn
from functools import partial

In [ ]:
torchvision.models.list_models()

In [ ]:
list(torchvision.models.get_model_weights("convnext_large"))

In [ ]:
weights = torchvision.models.ConvNeXt_Large_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_large(weights=weights).to(device)

In [ ]:
import torchvision.transforms.v2 as T

transforms = weights.transforms()
# transforms = T.Compose([
#     T.RandomHorizontalFlip(p=0.5),
#     T.RandomRotation(degrees=30),
#     T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
#     T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
#     T.ToImage(),
#     T.ToDtype(torch.float32, scale=True),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

In [ ]:
print(transforms)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    ordinal_encoder = LabelEncoder()
    encoded_labels = ordinal_encoder.fit_transform(labels)
    class_names = ordinal_encoder.classes_
    return encoded_labels, class_names


In [ ]:
encoded_labels, class_names = encode_labels(labels)

In [ ]:
print(class_names)

In [ ]:
print(encoded_labels)

In [ ]:
# from torch.utils.data import Dataset, DataLoader
# from PIL import Image
# from sklearn.model_selection import StratifiedShuffleSplit
# from torch.utils.data import Subset

# class MyImages(Dataset):
#     def __init__(self, paths, encoded_labels, split=None, transform=None):
#         self.paths = paths
#         self.encoded_labels = encoded_labels
#         self.transform = transform

#         sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
#         train_idx, test_idx = next(sss.split(self.paths, self.encoded_labels))
#         self.train_set = Subset(self.paths, train_idx)
#         self.test_set = Subset(self.paths, test_idx)

#     def __call__(self, split=None, transform=None):
#         self.transform = transform
#         if split == "train":
#             return self.train_set
#         elif split == "test":
#             return self.test_set

#     def __len__(self):
#         return len(self.paths)

#     def __getitem__(self, idx):
#         img = Image.open(self.paths[idx]).convert("RGB")
#         if self.transform:
#             img = self.transform(img)
#         return img, self.encoded_labels[idx]  # return path too (optional)


In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image
import numpy as np

class MyImages(Dataset):
    def __init__(self, paths, encoded_labels, split="train", test_size=0.2, val_size=0.1, random_state=42, transform=None,
                 split_indices=None):

        if split not in {None, "train", "val", "test"}:
            raise ValueError(f"split must be one of {{'None','train','val','test'}}, got {split!r}")

        self.transform = transform

        paths = np.array(list(paths))
        encoded_labels = np.array(list(encoded_labels))

        n = len(paths)
        if n != len(encoded_labels):
            raise ValueError(f"paths and encoded_labels must have same length, got {n} and {len(labels)}")

        if split_indices is None:
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
            trainval_idx, test_idx = next(sss1.split(paths, encoded_labels))

            trainval_fraction = 1.0 - test_size
            val_within_trainval = val_size / trainval_fraction

            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_trainval, random_state=random_state)

            trainval_paths = paths[trainval_idx]
            trainval_labels = encoded_labels[trainval_idx]

            train_rel_idx, val_rel_idx = next(sss2.split(trainval_paths, trainval_labels))
            train_idx = trainval_idx[train_rel_idx]
            val_idx = trainval_idx[val_rel_idx]

            split_indices = {"train": train_idx, "val": val_idx, "test": test_idx}

        idx = split_indices[split]

        self.paths = paths[idx].tolist()
        self.encoded_labels = encoded_labels[idx].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.encoded_labels[idx]


In [ ]:
from functools import partial

DefaultCloudImages = partial(
    MyImages,
    paths=paths,
    encoded_labels=encoded_labels,
    transform=transforms,
    test_size=0.2,
    val_size=0.1,
    random_state=42
)

In [ ]:
train_set = DefaultCloudImages(split="train")
valid_set = DefaultCloudImages(split="val")
test_set = DefaultCloudImages(split="test")

In [ ]:
train_set[0]

In [ ]:
len(train_set)
len(valid_set)
len(test_set)

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.CenterCrop(500),
])

# create a dataset instance for the train split (use MyImages to request a split)
clouds_to_display = DefaultCloudImages(split="train", transform=transforms)


In [ ]:
def plot_image(image):
    plt.imshow(image.permute(1, 2, 0))
    plt.axis("off")

In [ ]:
sample_clouds = sorted({y: img for img, y in clouds_to_display}.items())[:11]

plt.figure(figsize=(10, 6))
for class_id, image in sample_clouds:
    if class_id == 11: break
    plt.subplot(3, 4, class_id + 1)
    plot_image(image)
    plt.title(f"{class_id}: {class_names[class_id]}", fontsize=11)

plt.show()

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [ ]:
[name for name, child in model.named_children()]

In [ ]:
model.classifier

In [ ]:
n_classes = 11
model.classifier[2] = nn.Linear(1536, n_classes).to(device)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [ ]:
n_epochs = 5
optimizer = torch.optim.AdamW(model.parameters())
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=11).to(device)
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

In [ ]:
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

In [ ]:
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

In [ ]:
for param in model.parameters():
    param.requires_grad = True

In [ ]:
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

In [ ]:
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

In [ ]:
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

In [ ]:
evaluate_tm(model, test_loader, accuracy)

In [ ]:
import torchvision.transforms.v2 as T

transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)